# IEMOCAP Audio + Transcript Fusion Pipeline

This notebook combines the pooled Wave2Vec2 audio features and pooled DistilBERT transcript embeddings into one fused representation. It keeps the same basic bricks as the existing notebooks, but adds a proper validation split inside the training sessions and early stopping before the final evaluation on `Ses05`.


## Imports

In [ ]:
import ctypes
import os
import warnings

prefix = os.environ.get("CONDA_PREFIX")
if prefix:
    ctypes.CDLL(f"{prefix}/lib/libgcc_s.so.1", mode=ctypes.RTLD_GLOBAL)
    ctypes.CDLL(f"{prefix}/lib/libstdc++.so.6", mode=ctypes.RTLD_GLOBAL)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import torch
import torch.nn as nn
from sklearn.metrics import classification_report, confusion_matrix
from transformers import AutoModel, AutoTokenizer, Wav2Vec2Model, Wav2Vec2Processor

from iemocap_pipeline_utils import (
    EmotionClassifier,
    build_iemocap_metadata,
    build_label_maps,
    compute_class_weights,
    create_train_val_test_dataloaders,
    embed_texts_with_distilbert,
    encode_labels,
    extract_wave2vec_features_from_rows,
    filter_iemocap_metadata,
    load_iemocap_train_split,
    scale_feature_splits,
    train_model,
    train_val_test_split,
    evaluate_model,
)

warnings.filterwarnings("ignore")
np.random.seed(42)
torch.manual_seed(42)


## Configuration

In [ ]:
DATASET_ID = "AbstractTTS/IEMOCAP"
LABEL_COLUMN = "major_emotion"
DROP_LABELS = {"other"}
MIN_SAMPLES_PER_CLASS = 30
TEST_SESSIONS = ("Ses05",)
VALIDATION_SIZE = 0.15
RANDOM_STATE = 42
MAX_SAMPLES = None  # Set an int for quicker experiments.

WAVE2VEC_MODEL_NAME = "facebook/wav2vec2-base-960h"
TEXT_MODEL_NAME = "distilbert-base-uncased"
BATCH_SIZE = 32
NUM_EPOCHS = 50
LEARNING_RATE = 3e-4
DROPOUT = 0.3
EARLY_STOPPING_PATIENCE = 8
MONITOR = "loss"

CACHE_DIR = "IEMOCAP_cache"
AUDIO_FEATURE_CACHE_PATH = os.path.join(CACHE_DIR, "audio_wave2vec_features.npz")
TEXT_EMBEDDING_CACHE_PATH = os.path.join(CACHE_DIR, "provided_transcript_distilbert_embeddings.npz")
CHECKPOINT_PATH = os.path.join("models", "IEMOCAP_audio_transcript_fusion.pth")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device


## Load And Filter IEMOCAP

In [ ]:
iemocap_dataset = load_iemocap_train_split(DATASET_ID)
metadata = build_iemocap_metadata(iemocap_dataset)
filtered_metadata, label_counts = filter_iemocap_metadata(
    metadata,
    label_column=LABEL_COLUMN,
    drop_labels=DROP_LABELS,
    min_samples_per_class=MIN_SAMPLES_PER_CLASS,
    max_samples=MAX_SAMPLES,
    random_state=RANDOM_STATE,
)

texts = filtered_metadata["transcription"].fillna("").astype(str).str.strip().tolist()

print(f"Original samples: {len(metadata)}")
print(f"Filtered samples: {len(filtered_metadata)}")
print(f"Labels kept: {label_counts.index.tolist()}")
label_counts


In [ ]:
plt.figure(figsize=(10, 4))
sns.barplot(x=label_counts.index, y=label_counts.values, color="steelblue")
plt.title("IEMOCAP Label Distribution After Filtering")
plt.xlabel("Label")
plt.ylabel("Count")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

pd.crosstab(filtered_metadata["session_id"], filtered_metadata[LABEL_COLUMN])


## Build Audio And Text Features

In [ ]:
print(f"Loading Wave2Vec2 model: {WAVE2VEC_MODEL_NAME}")
processor = Wav2Vec2Processor.from_pretrained(WAVE2VEC_MODEL_NAME)
wave2vec_model = Wav2Vec2Model.from_pretrained(WAVE2VEC_MODEL_NAME).to(device)
wave2vec_model.eval()

audio_X = extract_wave2vec_features_from_rows(
    iemocap_dataset,
    filtered_metadata["row_idx"].to_numpy(),
    processor,
    wave2vec_model,
    device,
    cache_path=AUDIO_FEATURE_CACHE_PATH,
)

audio_X.shape


In [ ]:
print(f"Loading text model: {TEXT_MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(TEXT_MODEL_NAME)
text_model = AutoModel.from_pretrained(TEXT_MODEL_NAME).to(device)
text_model.eval()

text_X = embed_texts_with_distilbert(
    texts,
    tokenizer,
    text_model,
    device,
    cache_path=TEXT_EMBEDDING_CACHE_PATH,
    batch_size=16,
)

text_X.shape


In [ ]:
X = np.concatenate([audio_X, text_X], axis=1).astype(np.float32)
print(f"Audio feature shape: {audio_X.shape}")
print(f"Text feature shape: {text_X.shape}")
print(f"Fused feature shape: {X.shape}")


## Prepare Train / Validation / Test Split

In [ ]:
label_names, label_to_idx, idx_to_label = build_label_maps(filtered_metadata[LABEL_COLUMN])
y = encode_labels(filtered_metadata[LABEL_COLUMN], label_to_idx)

train_mask, val_mask, test_mask = train_val_test_split(
    filtered_metadata,
    label_column=LABEL_COLUMN,
    test_sessions=TEST_SESSIONS,
    validation_size=VALIDATION_SIZE,
    random_state=RANDOM_STATE,
)

train_metadata = filtered_metadata.loc[train_mask].reset_index(drop=True)
val_metadata = filtered_metadata.loc[val_mask].reset_index(drop=True)
test_metadata = filtered_metadata.loc[test_mask].reset_index(drop=True)

X_train_raw = X[train_mask]
X_val_raw = X[val_mask]
X_test_raw = X[test_mask]
y_train = y[train_mask]
y_val = y[val_mask]
y_test = y[test_mask]

X_train, X_val, X_test, scaler = scale_feature_splits(X_train_raw, X_val_raw, X_test_raw)
train_loader, val_loader, test_loader = create_train_val_test_dataloaders(
    X_train, y_train, X_val, y_val, X_test, y_test, batch_size=BATCH_SIZE
)
class_weights, class_counts = compute_class_weights(y_train, num_classes=len(label_names))

print(f"Train sessions: {sorted(train_metadata["session_id"].unique())}")
print(f"Validation sessions: {sorted(val_metadata["session_id"].unique())}")
print(f"Test sessions: {sorted(test_metadata["session_id"].unique())}")
print(f"Train shape: {X_train.shape}")
print(f"Validation shape: {X_val.shape}")
print(f"Test shape: {X_test.shape}")

split_counts = pd.DataFrame({
    "train": train_metadata[LABEL_COLUMN].value_counts().reindex(label_names, fill_value=0),
    "val": val_metadata[LABEL_COLUMN].value_counts().reindex(label_names, fill_value=0),
    "test": test_metadata[LABEL_COLUMN].value_counts().reindex(label_names, fill_value=0),
})
split_counts


## Train Fusion Classifier

In [ ]:
model = EmotionClassifier(
    input_size=X_train.shape[1],
    num_emotions=len(label_names),
    dropout_rate=DROPOUT,
).to(device)

history = train_model(
    model,
    train_loader,
    val_loader,
    device,
    num_epochs=NUM_EPOCHS,
    learning_rate=LEARNING_RATE,
    class_weights=class_weights,
    eval_name="val",
    early_stopping_patience=EARLY_STOPPING_PATIENCE,
    restore_best=True,
    monitor=MONITOR,
)
history


## Evaluate On Ses05

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(history["train_loss"], label="train")
axes[0].plot(history["val_loss"], label="val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_acc"], label="train")
axes[1].plot(history["val_acc"], label="val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()


In [ ]:
criterion = nn.CrossEntropyLoss(weight=class_weights.to(device))
test_loss, test_acc, all_preds, all_labels = evaluate_model(model, test_loader, criterion, device)

print(f"Test accuracy: {test_acc:.2f}%")
print(f"Test loss: {test_loss:.4f}")
print()
print(classification_report(all_labels, all_preds, target_names=label_names, zero_division=0))

cm = confusion_matrix(all_labels, all_preds)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=label_names, yticklabels=label_names)
plt.title("Confusion Matrix - IEMOCAP Audio + Transcript Fusion")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()


## Save Checkpoint

In [ ]:
os.makedirs(os.path.dirname(CHECKPOINT_PATH), exist_ok=True)
torch.save(
    {
        "model_state_dict": model.state_dict(),
        "label_names": label_names,
        "label_to_idx": label_to_idx,
        "idx_to_label": idx_to_label,
        "scaler_mean": scaler.mean_,
        "scaler_scale": scaler.scale_,
        "config": {
            "dataset_id": DATASET_ID,
            "label_column": LABEL_COLUMN,
            "drop_labels": sorted(DROP_LABELS),
            "min_samples_per_class": MIN_SAMPLES_PER_CLASS,
            "test_sessions": list(TEST_SESSIONS),
            "validation_size": VALIDATION_SIZE,
            "random_state": RANDOM_STATE,
            "wave2vec_model_name": WAVE2VEC_MODEL_NAME,
            "text_model_name": TEXT_MODEL_NAME,
            "learning_rate": LEARNING_RATE,
            "dropout": DROPOUT,
            "num_epochs": NUM_EPOCHS,
            "early_stopping_patience": EARLY_STOPPING_PATIENCE,
            "monitor": MONITOR,
        },
        "history": history,
    },
    CHECKPOINT_PATH,
)
print(f"Saved checkpoint to: {CHECKPOINT_PATH}")
